MODEL

In [5]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
import torch.optim as optim
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import numpy as np
from PIL import Image
import os

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
data_dir = "../data/processed"
batch_size = 32
num_epochs = 20
learning_rate = 0.0001
img_size = 224

In [20]:
transform_old = transforms.Compose([
    #transforms.Grayscale(num_output_channels=1),
    transforms.Resize((240, 180)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  
])

transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],  
                         [0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(
    root=data_dir,
    transform=transform
)

# Split into train and validation (80/20)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

class_names = train_dataset.dataset.classes
print("Ilość klass:", class_names)
print("Liczba klass:", len(dataset.classes))


Ilość klass: ['ALFA ROMEO', 'ALPINE', 'ARIEL', 'ASTON MARTIN', 'AUDI', 'BENTLEY', 'BMW', 'BUGATTI', 'CADILLAC', 'CHEVROLET', 'CHRYSLER', 'CITROEN', 'CORVETTE', 'DACIA', 'DAEWOO', 'DAIHATSU', 'DAIMLER', 'DAX', 'DODGE', 'DS AUTOMOBILES', 'FERRARI', 'FIAT', 'FORD', 'GMC', 'GREAT WALL', 'HONDA', 'HUMMER', 'HYUNDAI', 'INFINITI', 'ISUZU', 'IVECO', 'JAGUAR', 'JEEP', 'KIA', 'LAMBORGHINI', 'LANCIA', 'LAND ROVER', 'LEXUS', 'LINCOLN', 'LOTUS', 'MASERATI', 'MAYBACH', 'MAZDA', 'MCLAREN', 'MERCEDES-BENZ', 'MG', 'MINI', 'MITSUBISHI', 'MORGAN', 'MORRIS', 'NISSAN', 'PEUGEOT', 'POLESTAR', 'PONTIAC', 'PORSCHE', 'PROTON', 'RELIANT', 'RENAULT', 'ROLLS-ROYCE', 'ROVER', 'SAAB', 'SEAT', 'SKODA', 'SMART', 'SSANGYONG', 'SUBARU', 'SUZUKI', 'TESLA', 'TOYOTA', 'TRIUMPH', 'TVR', 'VOLKSWAGEN', 'VOLVO']
Liczba klass: 73


In [21]:
model = models.mobilenet_v2(pretrained=True)

for param in model.features.parameters():
    param.requires_grad = False

num_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, len(class_names))
)

model = model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=learning_rate)

for epoch in range(0):
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        batch_size = imgs.shape[0]
        out = model(imgs)
        loss = loss_fn(out, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print("Epoch: %d, Loss: %f" % (epoch, float(loss)))


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
num_epochs = 30

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    model.train()

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

    if(loss.item()<2.2):
        break
    print(f"Loss {loss:.4f}")

torch.save(model.state_dict(), "model.pt")
print("Model zapisany")


Epoch 1/30
Loss 3.8342

Epoch 2/30
Loss 3.7771

Epoch 3/30
Loss 4.1467

Epoch 4/30
Loss 3.8220

Epoch 5/30
Loss 3.4773

Epoch 6/30
Loss 3.3581

Epoch 7/30
Loss 2.8190

Epoch 8/30
Loss 2.8849

Epoch 9/30
Loss 3.1146

Epoch 10/30
Loss 2.2898

Epoch 11/30
Loss 3.2200

Epoch 12/30
Loss 2.8608

Epoch 13/30
Loss 2.5067

Epoch 14/30
Loss 3.2659

Epoch 15/30
Loss 2.6411

Epoch 16/30
Loss 2.8986

Epoch 17/30
Loss 2.9666

Epoch 18/30
Loss 3.2462

Epoch 19/30
Loss 3.5846

Epoch 20/30
Loss 2.7510

Epoch 21/30
Loss 3.3984

Epoch 22/30
Loss 3.4136

Epoch 23/30
Loss 2.8181

Epoch 24/30
Loss 3.9775

Epoch 25/30
Loss 3.9296

Epoch 26/30
✅ Model saved as 'car_brand_classifier_mobilenetv2.pt'


In [23]:
correct = 0
total = 0

with torch.no_grad():
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        batch_size = imgs.shape[0]
        outputs = model(imgs)
        _, predicted = torch.max(outputs, dim=1)
        total += labels.shape[0]
        correct += int((predicted == labels).sum())

print("Celność dla danych treningowych: %f", correct / total)

correct = 0
total = 0

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        batch_size = imgs.shape[0]
        outputs = model(imgs)
        _, predicted = torch.max(outputs, dim=1)
        total += labels.shape[0]
        correct += int((predicted == labels).sum())

print("Celność dla walidacji: %f", correct / total)

Celność dla danych treningowych: %f 0.4540370976541189
Celność dla walidacji: %f 0.3047982551799346
